# Time Bounds

Tests `storage_entry_start_time` / `storage_entry_end_time` parameters.
Streams spanning boundaries are cut; continuous stream mass is proportionally distributed.

In [1]:
import datetime
import os

%load_ext autoreload
%autoreload 2

from ethos_penalps.stream import StreamType
from ethos_penalps.testing.storage.storage_test_case import (
    StorageTestCaseSpecification,
    build_case_from_parameters,
)
from ethos_penalps.testing.storage.storage_test_case_io import save_storage_test_case
from ethos_penalps.testing.stream.stream_group_test_case import (
    StreamGroupTestCaseSpecification,
)
from ethos_penalps.testing.stream.stream_test_case import (
    BatchStreamStateParams as B,
)
from ethos_penalps.testing.stream.stream_test_case import (
    BatchStreamTestCaseSpecification,
    ContinuousStreamTestCaseSpecification,
)
from ethos_penalps.testing.stream.stream_test_case import (
    ContinuousStreamStateParams as C,
)

CASES_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "cases")
T0 = datetime.datetime(2024, 1, 1)

## Cut startup — continuous streams

Input runs T0-2h to T0+4h at 30 kg/h (180 kg), output runs T0-2h to T0+4h at 10 kg/h (60 kg).
Without the cut, storage rises from 0 to 120 over 6h (net +20 kg/h).
`storage_entry_start_time=T0` cuts the first 2h, removing 40 kg of net input.
Visible window T0 to T0+4h: storage rises from 0 to 80.

In [2]:
params = StorageTestCaseSpecification(
    name="cut_startup_continuous",
    description="Input 30 kg/h, output 10 kg/h, both T0-2h to T0+4h. start_time=T0 cuts startup.",
    start_time=T0,
    start_storage_level="auto_offset",
    input_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=180, rate=30, duration_seconds=21600)],
                start_offset_seconds=-7200,
            ),
        ]
    ),
    output_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=60, rate=10, duration_seconds=21600)],
                start_offset_seconds=-7200,
            ),
        ]
    ),
    storage_entry_start_time=T0,
)

case = build_case_from_parameters(params)
case.print_overview()

net_mass = float(case.duration_array[:, 3].sum())
print(f"Net mass in window: {net_mass:.1f} (expected 80.0)")

save_storage_test_case(case, CASES_DIR)

'cut_startup_continuous', Entries: 1, In: 1 ContinuousStream, Out: 1 ContinuousStream, Start level: 0.0, Min level: 0.0, End level: 80.0
Net mass in window: 80.0 (expected 80.0)
Saved case 'cut_startup_continuous' to /fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_startup_continuous/
  expected_storage_entries.csv             0.1 KB
  input_streams.csv                        0.1 KB
  output_streams.csv                       0.1 KB
  parameters.json                          0.8 KB


'/fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_startup_continuous'

## Cut cooldown — continuous streams

Input runs T0 to T0+6h at 10 kg/h (60 kg), output T0+1h to T0+7h at 30 kg/h (180 kg).
Full profile: rises 10 kg in first hour, then drops 20 kg/h for 5h, then drops 30 kg/h for 1h.
`storage_entry_end_time=T0+5h` cuts the last 2h of cooldown.
Visible window T0 to T0+5h: storage rises then falls, ending at -70.

In [3]:
params = StorageTestCaseSpecification(
    name="cut_cooldown_continuous",
    description="Input 10 kg/h T0 to T0+6h, output 30 kg/h T0+1h to T0+7h. end_time=T0+5h cuts cooldown.",
    start_time=T0,
    start_storage_level="auto_offset",
    input_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=60, rate=10, duration_seconds=21600)],
            ),
        ]
    ),
    output_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=180, rate=30, duration_seconds=21600)],
                start_offset_seconds=3600,
            ),
        ]
    ),
    storage_entry_end_time=T0 + datetime.timedelta(hours=5),
)

case = build_case_from_parameters(params)
case.print_overview()
save_storage_test_case(case, CASES_DIR)

'cut_cooldown_continuous', Entries: 2, In: 1 ContinuousStream, Out: 1 ContinuousStream, Start level: 70.0, Min level: 70.0, End level: 0.0
Saved case 'cut_cooldown_continuous' to /fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_cooldown_continuous/
  expected_storage_entries.csv             0.2 KB
  input_streams.csv                        0.1 KB
  output_streams.csv                       0.1 KB
  parameters.json                          0.8 KB


'/fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_cooldown_continuous'

## Cut both startup and cooldown

Input T0-2h to T0+8h at 30 kg/h (300 kg), output T0-1h to T0+9h at 10 kg/h (100 kg).
Full profile: input-only for 1h (+30), then both active for 9h (+20/h), then output-only for 1h (-10).
Window T0 to T0+6h cuts 2h of startup and 2h of cooldown.
Visible window shows storage rising at +20 kg/h from 0 to 120.

In [4]:
params = StorageTestCaseSpecification(
    name="cut_startup_and_cooldown",
    description="Input 30 kg/h T0-2h to T0+8h, output 10 kg/h T0-1h to T0+9h. Window T0 to T0+6h.",
    start_time=T0,
    start_storage_level="auto_offset",
    input_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=300, rate=30, duration_seconds=36000)],
                start_offset_seconds=-7200,
            ),
        ]
    ),
    output_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=100, rate=10, duration_seconds=36000)],
                start_offset_seconds=-3600,
            ),
        ]
    ),
    storage_entry_start_time=T0,
    storage_entry_end_time=T0 + datetime.timedelta(hours=6),
)

case = build_case_from_parameters(params)
case.print_overview()
save_storage_test_case(case, CASES_DIR)

'cut_startup_and_cooldown', Entries: 1, In: 1 ContinuousStream, Out: 1 ContinuousStream, Start level: 0.0, Min level: 0.0, End level: 120.0
Saved case 'cut_startup_and_cooldown' to /fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_startup_and_cooldown/
  expected_storage_entries.csv             0.1 KB
  input_streams.csv                        0.1 KB
  output_streams.csv                       0.1 KB
  parameters.json                          0.9 KB


'/fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_startup_and_cooldown'

## Cut startup with mixed streams

Continuous input from T0-2h to T0+4h (cut at startup). Batch outputs inside window (not cut).

In [5]:
params = StorageTestCaseSpecification(
    name="cut_startup_mixed_streams",
    description="Continuous input cut at startup, batch outputs inside window.",
    start_time=T0,
    start_storage_level="auto_offset",
    input_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=120, rate=20, duration_seconds=21600)],
                start_offset_seconds=-7200,
            ),
        ]
    ),
    output_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            BatchStreamTestCaseSpecification(
                states=[
                    B(mass=60, gap_seconds=3600),
                    B(mass=60, gap_seconds=7200),
                ],
                start_offset_seconds=3600,
                batch_delay_seconds=3600,
                max_batch_mass=100,
            ),
        ]
    ),
    storage_entry_start_time=T0,
)

case = build_case_from_parameters(params)
case.print_overview()
save_storage_test_case(case, CASES_DIR)

'cut_startup_mixed_streams', Entries: 5, In: 1 ContinuousStream, Out: 1 BatchStream, Start level: 100.0, Min level: 0.0, End level: 60.0
Saved case 'cut_startup_mixed_streams' to /fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_startup_mixed_streams/
  expected_storage_entries.csv             0.3 KB
  input_streams.csv                        0.1 KB
  output_streams.csv                       0.2 KB
  parameters.json                          0.8 KB


'/fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/cut_startup_mixed_streams'

## Multi-stream cut

Two inputs + two outputs. First input/output start before window (startup cut), second pair ends after (cooldown cut).

In [6]:
params = StorageTestCaseSpecification(
    name="multi_stream_cut",
    description="Two inputs + two outputs, both bounds cut through overlapping streams.",
    start_time=T0,
    start_storage_level="auto_offset",
    input_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=80, rate=20, duration_seconds=14400)],
                start_offset_seconds=-3600,
            ),
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=80, rate=20, duration_seconds=14400)],
                start_offset_seconds=10800,
            ),
        ]
    ),
    output_stream_group=StreamGroupTestCaseSpecification(
        stream_specifications=[
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=80, rate=20, duration_seconds=14400)],
                start_offset_seconds=-7200,
            ),
            ContinuousStreamTestCaseSpecification(
                states=[C(mass=80, rate=20, duration_seconds=14400)],
                start_offset_seconds=14400,
            ),
        ]
    ),
    storage_entry_start_time=T0,
    storage_entry_end_time=T0 + datetime.timedelta(hours=6),
)

case = build_case_from_parameters(params)
case.print_overview()
save_storage_test_case(case, CASES_DIR)

'multi_stream_cut', Entries: 4, In: 2 ContinuousStream, Out: 2 ContinuousStream, Start level: 0.0, Min level: 0.0, End level: 40.0
Saved case 'multi_stream_cut' to /fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/multi_stream_cut/
  expected_storage_entries.csv             0.3 KB
  input_streams.csv                        0.2 KB
  output_streams.csv                       0.2 KB
  parameters.json                          1.2 KB


'/fast/home/j-belina/ethos_penalps/test/generate/storage/06_time_bounds/cases/multi_stream_cut'